In [1]:
import os
import os.path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from datetime import datetime
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
import base64
sns.set()

In [2]:
datadir = "data"
data = os.path.join(datadir, "nationwide-encounters-fy21-fy24-state.csv")
df_cbp = pd.read_csv(data)

In [3]:
df_cbp["Demographic"] = df_cbp["Demographic"].str.replace("FMUA", "Individuals in a Family Unit (FMUA)")
df_cbp["Demographic"] = df_cbp["Demographic"].str.replace("UC / Single Minors", "Unaccompanied Children / Single Minors")

In [4]:
len(df_cbp["Citizenship"].unique())

22

In [5]:
encounter_group = df_cbp.groupby(["Land Border Region", "State", "Demographic", "Citizenship"]).sum().reset_index().sort_values(by = "Encounter Count", ascending = False)

In [6]:
encounter_group

,Land Border Region,State,Demographic,Citizenship,Fiscal Year,Month Grouping,Month (abbv),Title of Authority,Encounter Count
2234,Southwest Land Border,TX,Single Adults,MEXICO,161780,FYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFY...,APRAPRAUGAUGDECDECFEBFEBJANJANJULJULJUNJUNMARM...,Title 42Title 8Title 42Title 8Title 42Title 8T...,844262
2015,Southwest Land Border,AZ,Single Adults,MEXICO,161780,FYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFY...,APRAPRAUGAUGDECDECFEBFEBJANJANJULJULJUNJUNMARM...,Title 42Title 8Title 42Title 8Title 42Title 8T...,551660
2094,Southwest Land Border,CA,Single Adults,MEXICO,161780,FYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFY...,APRAPRAUGAUGDECDECFEBFEBJANJANJULJULJUNJUNMARM...,Title 42Title 8Title 42Title 8Title 42Title 8T...,495808
2244,Southwest Land Border,TX,Single Adults,VENEZUELA,161780,FYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFY...,APRAPRAUGAUGDECDECFEBFEBJANJANJULJULJUNJUNMARM...,Title 42Title 8Title 42Title 8Title 42Title 8T...,371840
2211,Southwest Land Border,TX,Individuals in a Family Unit (FMUA),HONDURAS,161780,FYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFYTDFY...,APRAPRAUGAUGDECDECFEBFEBJANJANJULJULJUNJUNMARM...,Title 42Title 8Title 42Title 8Title 42Title 8T...,367127
...,...,...,...,...,...,...,...,...,...
1126,Other,LA,Unaccompanied Children / Single Minors,OTHER,2022,FYTD,SEP,Title 8,1
1127,Other,MA,Accompanied Minors,ECUADOR,2024,FYTD,JUN,Title 8,1
1129,Other,MA,Accompanied Minors,INDIA,2023,FYTD,DEC,Title 8,1
1130,Other,MA,Accompanied Minors,NICARAGUA,2024,FYTD,FEB,Title 8,1


In [12]:
fig_treemap = px.treemap(encounter_group, 
                         path = ["Citizenship",  "Demographic"], 
                         values = 'Encounter Count', 
                         custom_data = ["Citizenship", "State", "Demographic", "Encounter Count"],
                         color = "Citizenship",
                         #color_continuous_scale="pubu",
                         color_discrete_map = {
                             "MEXICO": "#012A4A" ,           
                             'GUATEMALA': "#01497C",                        
                             'HONDURAS': "#2A6F97",  
                             'VENEZUELA': "#468FAF",              
                             'OTHER': "#89C2D9",        
                             'CUBA': "#013A63",             
                             'HAITI': "#014F86",             
                             'NICARAGUA': "#2C7DA0",               
                             'COLOMBIA': "#61A5C2",                    
                             'ECUADOR': "#A9D6E5",
                             "EL SALVADOR": "#012A4A" ,           
                             'UKRAINE': "#01497C",                            
                             'INDIA': "#2A6F97",  
                             'PHILIPPINES': "#468FAF",              
                             'CHINA, PEOPLES REPUBLIC OF': "#89C2D9",        
                             'BRAZIL': "#013A63",             
                             'PERU': "#014F86",             
                             'CANADA': "#2C7DA0",               
                             'RUSSIA': "#61A5C2",                   
                             'TURKEY': "#A9D6E5",
                             "ROMANIA": "#012A4A" ,           
                             'MYANMAR (BURMA)': "#01497C"}
                        )

with open("logo/CBP-logo-blue-lettering.png", "rb") as img:
    encoded_image = base64.b64encode(img.read()).decode()

fig_treemap.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="paper", yref="paper",
        x=0.99, y=1,
        sizex=0.2, sizey=0.2,
        xanchor="right", yanchor="bottom"
    )
)

fig_treemap.update_traces(
    hovertemplate =
                "<b>Citizenship:</b><br>" +
                "<b>%{customdata[0]}</b><br><br>" +
                "State of Entry: %{customdata[1]}<br>" +
                "Demographic: %{customdata[2]}<br>" +
                "Count: %{customdata[3]}" +
                "<extra></extra>",
    
)

fig_treemap.update_layout(
    margin=dict(t=140, l=25, r=25, b=25), 
    font_family = "Helvetica",
    width = 1600,
    height = 900,
    title=dict(
        text = "<b>Apprehensions, Inadmissibles, and Expulsions <br>by Citizenship at U.S. Ports of Entry 2021 - 2024</b>",
        font = dict(
            size=28,
            family = "Helvetica"),
        x = 0.018
        ),
    annotations=[
        dict(
            text="<i>Source: U.S. Customs and Border Protection, FY21 - FY24 Nationwide Encounters by State</i>",  
            #x=0.034, 
            x=0, 
            y=1.03,  
            font=dict(size=16, family="Helvetica"),  
            showarrow=False 
        )
    ]
)

fig_treemap.write_html("interactive_plots/cbp_encounter_distribution.html")